# 04 · Baselines y ablaciones atómicas — foco Basal-like

Este cuaderno mantiene el endpoint predictivo de **cuatro clases**, porque la topología Basal-like se aprende por contraste frente a las otras tres. Sin embargo, todos los análisis biológicos y las métricas específicas se restringen a **Basal-like**.

Contratos ejecutables:

1. elastic-net, random forest y XGBoost usan exactamente los **7.131 genes conectados del prior**, en el mismo orden que FULL;
2. cada brazo entrenado cambia un único campo respecto de FULL;
3. `NO_SIGNED_WEIGHT` elimina solo el signo del peso escalar mediante `abs(w)` y no se presenta como eliminación de toda anotación relacionada con activación/inhibición;
4. el bloque externo permanece cerrado.

Las funciones viven en `intergate/notebook04_controls.py`. El análisis de estabilidad y la comparación modular ya pertenecen al Notebook 03 y no se duplican aquí. Jumping Knowledge no se ablate.

In [ ]:
from pathlib import Path
import sys

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'intergate').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Ejecute este notebook dentro de InterGATE_V8.')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_transport_protocol, assert_v8_protocol, prepare_runtime,
)
from intergate.transport import TransportConfig
from intergate.notebook04_controls import (
    Notebook04Options, audit_baseline_feature_universe,
    audit_single_component_ablations, build_trained_arm_specs,
    evaluate_predictive_baselines_loco_exact_universe,
    run_target_score_sensitivity,
    run_trained_single_component_ablations,
    save_notebook04_outputs, summarise_target_biology_ablations,
)

# Únicos controles de ejecución que normalmente se modifican.
TARGET_CLASS = 'Basal-like'
EXPECTED_CONNECTED_GENES = 7131
ABLATION_RUN_MODE = 'confirmatory'       # 'pilot' o 'confirmatory'
RUN_XGBOOST = True
RUN_SCORE_SENSITIVITY = True
XGBOOST_DEVICE = 'cpu'            # p. ej. 'cuda' con XGBoost compatible
XGBOOST_N_JOBS = -1

OPTIONS = Notebook04Options(
    target_class=TARGET_CLASS,
    expected_connected_genes=EXPECTED_CONNECTED_GENES,
    ablation_run_mode=ABLATION_RUN_MODE,
    run_xgboost=RUN_XGBOOST,
    run_score_sensitivity=RUN_SCORE_SENSITIVITY,
    xgboost_device=XGBOOST_DEVICE,
    xgboost_n_jobs=XGBOOST_N_JOBS,
)
OPTIONS.validate()
assert_v8_protocol(CFG)
assert_transport_protocol(CFG)
runtime = prepare_runtime(CFG, include_external=False)
assert runtime['external_opened'] is False

ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / 'INTERGATE_V8'
OUT = PROJECT_ROOT / 'results_v8' / '04_controls'
FIRST_SEED = int(CFG.STABILITY_SEEDS[0])
ABLATION_SEEDS = (
    (FIRST_SEED,) if ABLATION_RUN_MODE == 'pilot'
    else tuple(map(int, CFG.STABILITY_SEEDS))
)
BASE_TRANSPORT = TransportConfig.from_global(CFG)
print('Target biológico:', TARGET_CLASS)
print('Endpoint predictivo: cuatro clases')
print('Semillas de ablación:', ABLATION_SEEDS)

## 1. Auditoría del universo de entrada

Esta celda detiene la ejecución si el runtime, el prior, el bundle FULL y los baselines no comparten exactamente los mismos 7.131 genes y el mismo orden.

In [ ]:
feature_audit = audit_baseline_feature_universe(
    runtime,
    full_bundle_dir=RUN_ROOT / f'seed_{FIRST_SEED}',
    expected_connected_genes=EXPECTED_CONNECTED_GENES,
)
display(feature_audit)

## 2. Baselines predictivos LOCO

Los tres modelos resuelven el mismo problema multiclase que InterGATE. Se guardan macro-F1 y balanced accuracy, junto con precisión, recall, F1 y AUROC one-vs-rest de Basal-like. No existe selección de genes ni ajuste de hiperparámetros usando la cohorte retenida.

In [ ]:
baselines = evaluate_predictive_baselines_loco_exact_universe(
    runtime,
    seed=FIRST_SEED,
    target_class=TARGET_CLASS,
    run_xgboost=RUN_XGBOOST,
    xgboost_device=XGBOOST_DEVICE,
    xgboost_n_jobs=XGBOOST_N_JOBS,
)
display(
    baselines.groupby('model')[
        ['macro_f1', 'balanced_accuracy', 'target_f1', 'target_auc_ovr']
    ].agg(['mean', 'std'])
)

## 3. Contrato de ablación de un solo componente

Antes de entrenar, se comparan todos los campos de configuración y los tensores efectivos del prior. Cualquier brazo con cero o más de un cambio —salvo FULL— genera un error.

In [ ]:
ARM_SPECS = build_trained_arm_specs(BASE_TRANSPORT)
ablation_contract = audit_single_component_ablations(ARM_SPECS)
display(ablation_contract[[
    'arm', 'changed_component', 'changed_fields',
    'n_changed_fields', 'single_component_contract_passed',
]])
print('Corridas nuevas previstas:', (len(ARM_SPECS) - 1) * len(ABLATION_SEEDS))

## 4. Ablaciones entrenadas y red Basal-like

FULL se reutiliza. Cada brazo restante se reentrena desde el mismo protocolo y modifica únicamente el componente declarado. La comparación biológica se limita al grafo exact-K Basal-like de la misma semilla.

In [ ]:
trained_metrics = run_trained_single_component_ablations(
    specs=ARM_SPECS,
    seeds=ABLATION_SEEDS,
    artifacts_root=ARTIFACTS,
    target_class=TARGET_CLASS,
    run_mode=ABLATION_RUN_MODE,
)
target_metric = f'f1_{TARGET_CLASS}'
display(trained_metrics[[
    'arm', 'seed', 'macro_f1', target_metric,
    'delta_macro_f1_vs_full_same_seed',
    'delta_target_f1_vs_full_same_seed',
]])

target_biology = summarise_target_biology_ablations(
    specs=ARM_SPECS,
    seeds=ABLATION_SEEDS,
    artifacts_root=ARTIFACTS,
    target_class=TARGET_CLASS,
)
display(target_biology)

## 5. Sensibilidad post hoc del score Basal-like

Este bloque no es una ablación arquitectónica: mantiene los checkpoints y reconstruye únicamente el exact-K Basal-like eliminando, por separado, el término de contexto o el de soporte.

In [ ]:
score_summary = None
score_overlap = None
if RUN_SCORE_SENSITIVITY:
    score_summary, score_overlap = run_target_score_sensitivity(
        runtime,
        run_root=RUN_ROOT,
        output_dir=OUT / 'posthoc_score_sensitivity_Basal-like',
        seeds=tuple(map(int, CFG.STABILITY_SEEDS)),
        base_transport=BASE_TRANSPORT,
        target_class=TARGET_CLASS,
        cfg=CFG,
    )
    display(
        score_overlap.groupby('score_variant')[
            'jaccard_with_full_score'
        ].agg(['mean', 'std'])
    )

## 6. Guardado con nombres estables

In [ ]:
save_notebook04_outputs(
    output_dir=OUT,
    feature_audit=feature_audit,
    baselines=baselines,
    ablation_contract=ablation_contract,
    trained_metrics=trained_metrics,
    target_biology=target_biology,
    score_summary=score_summary,
    score_overlap=score_overlap,
)
print('Notebook 04 completo:', OUT)
print('El bloque externo permanece cerrado.')

In [ ]:
# ============================================================
# 7. Bloqueo post hoc del score biológico
#    No reentrena el modelo y no abre el externo
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from intergate.biology import aggregate_biology_seed_tables
from intergate.notebook04_controls import (
    _rebuild_target_biology_network,
)

LOCKED_CONTEXT_WEIGHT = 0.35
LOCKED_SUPPORT_WEIGHT = 0.075

LOCK_ROOT = (
    OUT
    / "locked_biology_score_w0075"
)
LOCK_SEED_ROOT = LOCK_ROOT / "seed_tables"
LOCK_CONSENSUS = LOCK_ROOT / "biology_consensus"

LOCK_ROOT.mkdir(parents=True, exist_ok=True)
LOCK_SEED_ROOT.mkdir(parents=True, exist_ok=True)
LOCK_CONSENSUS.mkdir(parents=True, exist_ok=True)

SEEDS_LOCKED = tuple(map(int, CFG.STABILITY_SEEDS))
CLASS_NAMES_LOCKED = tuple(map(str, runtime["classes"]))

if len(SEEDS_LOCKED) != 5:
    raise ValueError("El protocolo bloqueado requiere cinco semillas.")

if runtime["external_opened"]:
    raise RuntimeError("El bloque externo está abierto.")

def safe_name_locked(value):
    return "".join(
        char if char.isalnum() or char in "-_" else "_"
        for char in str(value)
    )

rebuild_rows = []

# Reconstrucción Exact-K por clase y semilla usando los artefactos densos
# ya calculados por el notebook 02.
for class_name in CLASS_NAMES_LOCKED:
    safe = safe_name_locked(class_name)

    for seed in SEEDS_LOCKED:
        run_dir = RUN_ROOT / f"seed_{seed}"

        table, summary, _ = _rebuild_target_biology_network(
            runtime,
            run_dir=run_dir,
            seed=seed,
            target_class=class_name,
            score_variant="LOCKED_W0075",
            context_weight=LOCKED_CONTEXT_WEIGHT,
            support_weight=LOCKED_SUPPORT_WEIGHT,
            cfg=CFG,
        )

        # La función de agregación espera este nombre.
        table = table.rename(
            columns={"transport_support": "selection_frequency"}
        )

        # Derivar K del grafo original; no fijarlo manualmente.
        original_file = run_dir / f"biology_edges_{safe}.csv"
        expected_k = len(pd.read_csv(original_file))

        if len(table) != expected_k:
            raise RuntimeError(
                f"{class_name}/seed={seed}: "
                f"Exact-K roto: {len(table)} != {expected_k}"
            )

        seed_output = LOCK_SEED_ROOT / f"seed_{seed}"
        seed_output.mkdir(parents=True, exist_ok=True)

        table.to_csv(
            seed_output / f"biology_edges_{safe}.csv",
            index=False,
        )

        summary.update({
            "locked_context_weight": LOCKED_CONTEXT_WEIGHT,
            "locked_support_weight": LOCKED_SUPPORT_WEIGHT,
            "expected_k": expected_k,
            "external_opened": False,
        })
        rebuild_rows.append(summary)

rebuild_summary = pd.DataFrame(rebuild_rows)
rebuild_summary.to_csv(
    LOCK_ROOT / "locked_per_seed_network_summary.csv",
    index=False,
)

# Consenso multisemilla Exact-K con la misma regla para las cuatro clases.
locked_consensus = aggregate_biology_seed_tables(
    LOCK_SEED_ROOT,
    seeds=SEEDS_LOCKED,
    class_names=CLASS_NAMES_LOCKED,
    output_dir=LOCK_CONSENSUS,
    pool_multiplier=CFG.BIOLOGY_MODULE_POOL_MULTIPLIER,
    target_components=CFG.BIOLOGY_MODULE_TARGET_COMPONENTS,
    min_component_edges=CFG.BIOLOGY_MODULE_MIN_EDGES,
)

# Núcleos 5/5:
# primario = genes incidentes en la misma arista dirigida retenida 5/5;
# secundario = genes presentes como nodos en las cinco redes.
edge_supported_cores = {}
node_presence_cores = {}

for class_name in CLASS_NAMES_LOCKED:
    safe = safe_name_locked(class_name)

    edge_sets = []
    gene_sets = []

    for seed in SEEDS_LOCKED:
        frame = pd.read_csv(
            LOCK_SEED_ROOT
            / f"seed_{seed}"
            / f"biology_edges_{safe}.csv"
        )

        selected_edges = set(zip(
            frame["source_gene"].astype(str).str.upper(),
            frame["target_gene"].astype(str).str.upper(),
        ))
        edge_sets.append(selected_edges)
        gene_sets.append({
            gene
            for edge in selected_edges
            for gene in edge
        })

    strict_edges = set.intersection(*edge_sets)
    strict_nodes = set.intersection(*gene_sets)

    edge_supported_cores[class_name] = sorted({
        gene for edge in strict_edges for gene in edge
    })
    node_presence_cores[class_name] = sorted(strict_nodes)

panels = {
    "protocol": {
        "target_class": TARGET_CLASS,
        "context_weight": LOCKED_CONTEXT_WEIGHT,
        "support_weight": LOCKED_SUPPORT_WEIGHT,
        "primary_definition": (
            "genes incident to an identical directed edge selected "
            "in all five locked Exact-K networks"
        ),
        "secondary_definition": (
            "genes present as nodes in all five locked Exact-K networks"
        ),
        "seeds": list(SEEDS_LOCKED),
        "retraining_performed": False,
        "known_signatures_used_for_selection": False,
        "external_opened": False,
    },
    "strict_5of5": edge_supported_cores,
    "edge_supported_gene_core_5of5": edge_supported_cores,
    "node_presence_core_5of5": node_presence_cores,
}

(LOCK_ROOT / "biology_panels_locked.json").write_text(
    json.dumps(panels, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

manifest = {
    "status": "locked_before_biological_and_external_validation",
    "target_class_for_weight_selection": TARGET_CLASS,
    "applied_identically_to_classes": list(CLASS_NAMES_LOCKED),
    "context_weight": LOCKED_CONTEXT_WEIGHT,
    "support_weight": LOCKED_SUPPORT_WEIGHT,
    "selection_basis": (
        "development-only structural trade-off between cohort-deletion "
        "support, five-seed reproducibility and network fragmentation"
    ),
    "source_artifacts": "transport_support_dense.npz",
    "training_repeated": False,
    "external_opened": False,
}

(LOCK_ROOT / "locked_score_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

locked_rows = []
for class_name, payload in locked_consensus.items():
    locked_rows.append({
        "class_name": class_name,
        "k_edges": payload["k_edges"],
        "strict_edges_5of5": payload["strict_edge_core"],
        "mean_pairwise_edge_jaccard":
            payload["mean_pairwise_edge_jaccard"],
        "consensus_nodes": payload["topology"]["n_nodes"],
        "consensus_components": payload["topology"]["n_components"],
        "consensus_giant_fraction":
            payload["topology"]["giant_fraction"],
    })

locked_table = pd.DataFrame(locked_rows)
display(locked_table)

print("Score biológico bloqueado correctamente.")
print("Peso de contexto:", LOCKED_CONTEXT_WEIGHT)
print("Peso de soporte:", LOCKED_SUPPORT_WEIGHT)
print("Salida:", LOCK_ROOT)
print("No se ha reentrenado ninguna semilla.")
print("El bloque externo permanece cerrado.")